<a href="https://colab.research.google.com/github/mariamhaider307-collab/FINMATE-AI/blob/main/Experiment_02_Verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import random

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Experiment 02 environment ready!")

Experiment 02 environment ready!


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

products = [
    "Toothpaste", "Shampoo", "Soap", "Rice", "Milk",
    "Biscuits", "Juice", "Toothbrush", "Detergent", "Coffee"
]

def generate_scenarios(n, start_id):
    data = []

    for i in range(n):
        remembered = np.random.randint(1, 11)
        memory_age = np.random.randint(1, 121)

        # Higher memory age means higher change probability
        change_probability = 0.02 + (memory_age / 120) * 0.78

        changed = np.random.binomial(1, change_probability)

        actual = remembered

        if changed:
            actual = np.random.choice(
                [x for x in range(1, 11) if x != remembered]
            )

        data.append({
            "Scenario_ID": start_id + i,
            "Product": np.random.choice(products),
            "Remembered_Location": remembered,
            "Actual_Location": actual,
            "Location_Changed": changed,
            "Memory_Age_Minutes": memory_age,
            "Communication_Delay": round(
                np.random.uniform(0.1, 2.0), 2
            )
        })

    return pd.DataFrame(data)

train_data = generate_scenarios(5000, 1)
test_data = generate_scenarios(5000, 5001)

print("Training:", len(train_data))
print("Testing:", len(test_data))
print("Training change rate:", round(train_data["Location_Changed"].mean() * 100, 2), "%")
print("Testing change rate:", round(test_data["Location_Changed"].mean() * 100, 2), "%")

Training: 5000
Testing: 5000
Training change rate: 41.02 %
Testing change rate: 43.16 %


In [ ]:
print("TRAINING DATA")
print(train_data["Location_Changed"].value_counts())
print("Change rate:", round(train_data["Location_Changed"].mean() * 100, 2), "%")

print("\nTESTING DATA")
print(test_data["Location_Changed"].value_counts())
print("Change rate:", round(test_data["Location_Changed"].mean() * 100, 2), "%")

TRAINING DATA
Location_Changed
0    2949
1    2051
Name: count, dtype: int64
Change rate: 41.02 %

TESTING DATA
Location_Changed
0    2842
1    2158
Name: count, dtype: int64
Change rate: 43.16 %


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Features available to the agent
features = [
    "Product",
    "Memory_Age_Minutes",
    "Communication_Delay"
]

# Prepare training and testing inputs
X_train = pd.get_dummies(train_data[features], columns=["Product"])
X_test = pd.get_dummies(test_data[features], columns=["Product"])

# Ensure both datasets have identical columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# The learning target: whether the remembered location is outdated
y_train = train_data["Location_Changed"]
y_test = test_data["Location_Changed"]

# Create and train the agent
agent = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=8
)

agent.fit(X_train, y_train)

# Evaluate on unseen testing data
predictions = agent.predict(X_test)

print("AGENT TRAINING COMPLETE")
print("Training scenarios:", len(X_train))
print("Testing scenarios:", len(X_test))
print("Prediction accuracy:", round(accuracy_score(y_test, predictions) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, predictions))

AGENT TRAINING COMPLETE
Training scenarios: 5000
Testing scenarios: 5000
Prediction accuracy: 69.18 %

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.78      0.74      2842
           1       0.66      0.58      0.62      2158

    accuracy                           0.69      5000
   macro avg       0.69      0.68      0.68      5000
weighted avg       0.69      0.69      0.69      5000



In [ ]:
import time
import pandas as pd

BASE_COLLECTION_TIME = 10.0

def evaluate_strategy(strategy, test_data, predictions=None):
    results = []

    for i, (_, row) in enumerate(test_data.iterrows()):

        start = time.perf_counter()

        if strategy == "Always Verify":
            verify = True

        elif strategy == "Reuse Memory":
            verify = False

        elif strategy == "Trained Agent":
            verify = bool(predictions[i])

        decision_time = time.perf_counter() - start

        communication_time = (
            row["Communication_Delay"] if verify else 0.0
        )

        if verify:
            selected_location = row["Actual_Location"]
        else:
            selected_location = row["Remembered_Location"]

        mistake = int(selected_location != row["Actual_Location"])

        total_time = (
            BASE_COLLECTION_TIME
            + communication_time
            + decision_time
        )

        results.append({
            "Scenario_ID": row["Scenario_ID"],
            "Strategy": strategy,
            "Verification": verify,
            "Selected_Location": selected_location,
            "Actual_Location": row["Actual_Location"],
            "Verification_Time": communication_time,
            "Computing_Time": decision_time,
            "Total_Collection_Time": total_time,
            "Mistake": mistake,
            "Success": 1 - mistake
        })

    return pd.DataFrame(results)


# Evaluate all three strategies
always_verify_results = evaluate_strategy(
    "Always Verify", test_data
)

reuse_memory_results = evaluate_strategy(
    "Reuse Memory", test_data
)

trained_agent_results = evaluate_strategy(
    "Trained Agent", test_data, predictions
)

# Export separate CSV files
always_verify_results.to_csv(
    "always_verify_results.csv", index=False
)

reuse_memory_results.to_csv(
    "reuse_memory_results.csv", index=False
)

trained_agent_results.to_csv(
    "trained_agent_results.csv", index=False
)

print("ALL THREE STRATEGIES TESTED!")
print("Three CSV files created successfully.")

ALL THREE STRATEGIES TESTED!
Three CSV files created successfully.


In [ ]:
all_results = pd.concat([
    always_verify_results,
    reuse_memory_results,
    trained_agent_results
])

summary = all_results.groupby("Strategy").agg(
    Verification_Requests=("Verification", "sum"),
    Communication_Time=("Verification_Time", "sum"),
    Computing_Time=("Computing_Time", "sum"),
    Average_Collection_Time=("Total_Collection_Time", "mean"),
    Total_Mistakes=("Mistake", "sum"),
    Success_Rate=("Success", "mean")
).reset_index()

summary["Success_Rate"] *= 100

display(summary.round(4))

summary.to_csv("experiment_02_summary.csv", index=False)

,Strategy,Verification_Requests,Communication_Time,Computing_Time,Average_Collection_Time,Total_Mistakes,Success_Rate
0,Always Verify,5000,5274.81,0.0018,11.0550,0,100.00
1,Reuse Memory,0,0.00,0.0020,10.0000,2158,56.84
2,Trained Agent,1879,1912.08,0.0051,10.3824,910,81.80


In [ ]:
import zipfile
from google.colab import files

csv_files = [
    "always_verify_results.csv",
    "reuse_memory_results.csv",
    "trained_agent_results.csv",
    "experiment_02_summary.csv"
]

with zipfile.ZipFile("Experiment_02_Results.zip", "w") as z:
    for filename in csv_files:
        z.write(filename)

files.download("Experiment_02_Results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
display(summary)

,Strategy,Verification_Requests,Communication_Time,Computing_Time,Average_Collection_Time,Total_Mistakes,Success_Rate
0,Always Verify,5000,5274.81,0.001820,11.054962,0,100.00
1,Reuse Memory,0,0.00,0.002003,10.000000,2158,56.84
2,Trained Agent,1879,1912.08,0.005058,10.382417,910,81.80


In [ ]:
print("Always Verify mistakes:",
      always_verify_results["Mistake"].sum())

print("Reuse Memory mistakes:",
      reuse_memory_results["Mistake"].sum())

print("Trained Agent mistakes:",
      trained_agent_results["Mistake"].sum())

Always Verify mistakes: 0
Reuse Memory mistakes: 2158
Trained Agent mistakes: 910


In [ ]:
print(always_verify_results.shape[0])
print(reuse_memory_results.shape[0])
print(trained_agent_results.shape[0])

5000
5000
5000
